# 1. Determinar que se va a clasifiar

Se hará una clasificación para predecir si un cliente esperará o no por una mesa. Usando como variable objetivo _class_

# 2. Análisis Exploratorio de los datos (EDA)

### Bibliotecas y carga inicial de datos

In [8]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

### Carga de datos

In [2]:
df = pd.read_csv('restaurantData.csv')

### Análisis inicial

In [18]:
# Información básica del dataset 

print(f"Dimensiones: {df.shape}")
print(f"\nNombres de las columnas:\n {df.columns} ")
print (f"\nTips de datos:\n {df.dtypes}")
print(f"\nDescripción de variables numéricas:\n {df.describe()}")
print(f"\nDescripción de variables categóricas:\n {df.describe(include=['object'])}")
print(f"\nValores únicos por columna:\n {df.nunique()}")
print(f"\nValores nulos por columna:\n {df.isnull().sum()}")


Dimensiones: (100, 11)

Nombres de las columnas:
 Index(['class', 'alternate', 'bar', 'fri/sat', 'hungry', 'patrons', 'price',
       'raining', 'reservation', 'type', ' waitestimate'],
      dtype='object') 

Tips de datos:
 class            object
alternate        object
bar              object
fri/sat          object
hungry           object
patrons          object
price            object
raining          object
reservation      object
type             object
 waitestimate     int64
dtype: object

Descripción de variables numéricas:
        waitestimate
count    100.000000
mean      25.000000
std       23.028309
min        0.000000
25%        7.500000
50%       20.000000
75%       37.500000
max       60.000000

Descripción de variables categóricas:
             class alternate  bar fri/sat hungry patrons price raining  \
count         100       100  100     100    100     100   100     100   
unique          2         2    2       2      2       3     3       2   
top      wontwait  

In [31]:
# Muestra de datos
df.head(10)

,class,alternate,bar,fri/sat,hungry,patrons,price,raining,reservation,type,waitestimate
0,wontwait,no,no,no,yes,none,DDD,no,yes,thai,0
1,willwait,yes,yes,no,yes,some,DDD,no,no,thai,0
2,willwait,yes,no,yes,yes,some,DD,no,no,thai,10
3,willwait,yes,no,no,yes,some,D,yes,no,italian,0
4,wontwait,no,no,no,no,none,DDD,no,no,french,60
5,wontwait,no,yes,yes,no,full,DDD,yes,no,burger,60
6,wontwait,yes,yes,no,yes,none,DDD,no,yes,french,60
7,wontwait,yes,yes,yes,no,full,DDD,no,no,thai,60
8,willwait,yes,no,yes,yes,some,DDD,no,yes,burger,10
9,wontwait,yes,yes,no,yes,none,DDD,no,yes,burger,10


### Visualización de datos

In [32]:
# Análisis de relación entre variables categóricas y la clase objetivo
categorical_vars = ['alternate', 'bar', 'fri/sat', 'hungry', 'patrons', 'price', 
                   'raining', 'reservation', 'type']

print(f"\n=== RELACIÓN CON VARIABLE OBJETIVO ===")
for var in categorical_vars:
    print(f"\n{var}:")
    cross_tab = pd.crosstab(df[var], df['class'], normalize='index') * 100
    print(cross_tab.round(2))


=== RELACIÓN CON VARIABLE OBJETIVO ===

alternate:
class      willwait  wontwait
alternate                    
no            52.94     47.06
yes           44.90     55.10

bar:
class  willwait  wontwait
bar                      
no        56.60     43.40
yes       40.43     59.57

fri/sat:
class    willwait  wontwait
fri/sat                    
no          42.31     57.69
yes         56.25     43.75

hungry:
class   willwait  wontwait
hungry                    
no         35.42     64.58
yes        61.54     38.46

patrons:
class    willwait  wontwait
patrons                    
full        47.06     52.94
none         0.00    100.00
some       100.00      0.00

price:
class  willwait  wontwait
price                    
D         58.82     41.18
DD        48.28     51.72
DDD       40.54     59.46

raining:
class    willwait  wontwait
raining                    
no          50.00     50.00
yes         47.83     52.17

reservation:
class        willwait  wontwait
reservation            

### Preprocesamiento de Datos

In [21]:
# Cambio de categorías con label encoding
columnas_object = df.select_dtypes(include=['object']).columns
df_encoded = df.copy()
df_encoded[columnas_object] = df[columnas_object].apply(LabelEncoder().fit_transform)

In [22]:
df_encoded.head()

,class,alternate,bar,fri/sat,hungry,patrons,price,raining,reservation,type,waitestimate
0,1,0,0,0,1,1,2,0,1,3,0
1,0,1,1,0,1,2,2,0,0,3,0
2,0,1,0,1,1,2,1,0,0,3,10
3,0,1,0,0,1,2,0,1,0,2,0
4,1,0,0,0,0,1,2,0,0,1,60


In [23]:
for columna in df.columns:
    orig_vals = df[columna].unique()
    encoded_vals = df_encoded[columna].unique()
    
    print(f"Columna: '{columna}'")
    print(f"\t-Original:   {orig_vals}")
    print(f"\t-Codificado: {encoded_vals}")
    print()


Columna: 'class'
	-Original:   [' wontwait' ' willwait']
	-Codificado: [1 0]

Columna: 'alternate'
	-Original:   [' no' ' yes']
	-Codificado: [0 1]

Columna: 'bar'
	-Original:   [' no' ' yes']
	-Codificado: [0 1]

Columna: 'fri/sat'
	-Original:   [' no' ' yes']
	-Codificado: [0 1]

Columna: 'hungry'
	-Original:   [' yes' ' no']
	-Codificado: [1 0]

Columna: 'patrons'
	-Original:   [' none' ' some' ' full']
	-Codificado: [1 2 0]

Columna: 'price'
	-Original:   [' DDD' ' DD' ' D']
	-Codificado: [2 1 0]

Columna: 'raining'
	-Original:   [' no' ' yes']
	-Codificado: [0 1]

Columna: 'reservation'
	-Original:   [' yes' ' no']
	-Codificado: [1 0]

Columna: 'type'
	-Original:   [' thai' ' italian' ' french' ' burger']
	-Codificado: [3 2 1 0]

Columna: ' waitestimate'
	-Original:   [ 0 10 60 30]
	-Codificado: [ 0 10 60 30]



# 3. Decisión de mejor modelo para clasificación

In [34]:
# Bilbiotecas para modelado
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier

# Bibliotecas usuales
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix


In [35]:
# División de datos de entrenamiento y de prueba
X_train, X_test, y_train, y_test = train_test_split(
    df_encoded.drop('class', axis=1),
    df_encoded['class'],
    test_size=0.3,
    random_state=7)

In [37]:
# Creación de modelos
models = {
    'Decision Tree': DecisionTreeClassifier(criterion='entropy'),
    'SVM': SVC(kernel='linear'),
    'KNN': KNeighborsClassifier(3),
    'Neural Network': MLPClassifier(hidden_layer_sizes=(10),activation='tanh',max_iter=1000,random_state=7)
}

In [42]:
# Entrenamiento de modelos
resultados = {}

for name, model in models.items():
    # Entrenamiento
    model.fit(X_train, y_train)
    
    # Predicción
    y_pred = model.predict(X_test)

    # Precisión del modelo
    accuracy = accuracy_score(y_test, y_pred)
    resultados[name] = {
        'model': model,
        'accuracy': accuracy,
        'predictions': y_pred
    }

    # Matriz de confusión
    cm = confusion_matrix(y_test, y_pred)
    
    print(f"\n=== {name} ===")
    print(f"Precisión de {name}: {accuracy:.4f}")
    print("Matriz de confusión:")
    print(cm)


=== Decision Tree ===
Precisión de Decision Tree: 0.9333
Matriz de confusión:
[[17  2]
 [ 0 11]]

=== SVM ===
Precisión de SVM: 0.7667
Matriz de confusión:
[[13  6]
 [ 1 10]]

=== KNN ===
Precisión de KNN: 0.6000
Matriz de confusión:
[[ 9 10]
 [ 2  9]]

=== Neural Network ===
Precisión de Neural Network: 0.6000
Matriz de confusión:
[[10  9]
 [ 3  8]]


s:\UAM\25-O\Aprendizaje Maquinal\Laboratorio\restaurante\venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [45]:
# Comparar modelos
for name, result in resultados.items():
    print(f"{name}: {result['accuracy']:.4f}")

Decision Tree: 0.9333
SVM: 0.7667
KNN: 0.6000
Neural Network: 0.6000


Con base en la precisión obtenida de cada uno de los modelos. Podemos determinar que el mejor modelo es el Árbol de Decisión, con una precisión del 93.33%

# 4. Resultados e interpretación

**¿Qué se desea obtener?**
Que el modelo permita predecir el comportamiento de clientes respecto a la espera por mesas.

**¿Qué tan preciso es el modelo?**
- Precisión general: 93.33%
- Matriz de confusión:
[[17, 2] -> 17 wontwait correctos, 2 incorrectos
[0, 11]] -> 0 will wait incorrectos, 11 correctos

**¿Qué acciones permite llevar a cabo el modelo?**
1. Gestión de colas en tiempo real
2. Asignación dinámica de personal
3. Optmización de reservas